# FairGate: Fairness-Gated MLOps Pipeline for Credit Default Prediction
**Raghav Bhatambarekar (23070126024) · Group 16 · MLOps**

A model only reaches production if it passes **both** a performance check and a **bias check**.
Predictions come with reason codes, and production traffic is monitored for **fairness drift** as well as data drift.

**Run it:** `Runtime → Run all`. Takes about 3 minutes on the synthetic sample.

## 1. Install dependencies

In [ ]:
!pip install -q fairlearn mlflow lightgbm fastapi httpx pyyaml pyarrow gradio

## 2. Write the project files
Each cell below creates one file of the pipeline.

In [ ]:
import os
os.makedirs('/content/fairgate/src', exist_ok=True)
os.makedirs('/content/fairgate/app', exist_ok=True)
os.makedirs('/content/fairgate/tests', exist_ok=True)
%cd /content/fairgate

In [ ]:
%%writefile params.yaml
data:
  raw_path: data/raw/application_train.csv   # Kaggle Home Credit file (or the synthetic sample)
  test_size: 0.25
  seed: 42

train:
  n_estimators: 400
  learning_rate: 0.05
  num_leaves: 31
  target_reject_rate: 0.15     # bank rejects the riskiest 15% of applicants

gate:
  sensitive: [gender, age_band]
  max_dp_diff: 0.05            # demographic parity: max gap in rejection rate between groups
  max_eo_diff: 0.10            # equal opportunity: max gap in true positive rate (defaulters caught)
  min_auc: 0.70                # performance floor
  mitigation_constraint: demographic_parity

monitor:
  n_batches: 6
  batch_size: 5000
  psi_limit: 0.2
  fair_limit: 0.08             # wider than the gate limit: small live batches are noisier


In [ ]:
%%writefile src/common.py
"""Shared config, feature definitions and fairness metrics used by every stage."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score
from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate

ROOT = Path(__file__).resolve().parents[1]
PARAMS = yaml.safe_load(open(ROOT / "params.yaml"))
PROC = ROOT / "data" / "processed"
MODELS = ROOT / "models"
REPORTS = ROOT / "reports"
for d in (ROOT / "data" / "raw", PROC, MODELS, REPORTS):
    d.mkdir(parents=True, exist_ok=True)

TARGET = "TARGET"
NUMERIC = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE", "DAYS_EMPLOYED",
           "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "REGION_POPULATION_RELATIVE",
           "CNT_CHILDREN", "CNT_FAM_MEMBERS", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "OWN_CAR_AGE"]
CATEGORICAL = ["NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
               "NAME_HOUSING_TYPE", "OCCUPATION_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY"]
ENGINEERED = ["credit_income_ratio", "annuity_income_ratio", "credit_term", "goods_credit_ratio"]
# Sensitive attributes are NOT model inputs; they are only used to measure (and mitigate) bias.
SENSITIVE_RAW = ["CODE_GENDER", "DAYS_BIRTH"]
FEATURES = NUMERIC + ENGINEERED + CATEGORICAL


def age_band(days_birth):
    age = -np.asarray(days_birth, dtype=float) / 365.25
    return np.where(age < 30, "<30", np.where(age <= 50, "30-50", ">50"))


def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)   # Home Credit placeholder value
    df["credit_income_ratio"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["annuity_income_ratio"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["credit_term"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    df["goods_credit_ratio"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    df["gender"] = df["CODE_GENDER"].astype(str)
    df["age_band"] = age_band(df["DAYS_BIRTH"])
    return df


def to_model_frame(df: pd.DataFrame, categories: dict) -> pd.DataFrame:
    X = df[FEATURES].copy()
    for c in NUMERIC + ENGINEERED:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype(float)
    for c in CATEGORICAL:
        X[c] = pd.Categorical(X[c].astype("object"), categories=categories[c])
    return X


def fairness_report(y_true, y_pred, y_score, sens: pd.DataFrame, gate_cfg: dict) -> dict:
    """y_pred = 1 means REJECTED (predicted default). Returns metrics + pass/fail."""
    out = {"auc": float(roc_auc_score(y_true, y_score)), "reject_rate": float(np.mean(y_pred)), "groups": {}}
    worst_dp, worst_eo = 0.0, 0.0
    for col in gate_cfg["sensitive"]:
        mf = MetricFrame(metrics={"reject_rate": selection_rate, "tpr": true_positive_rate},
                         y_true=y_true, y_pred=y_pred, sensitive_features=sens[col])
        g = mf.by_group
        dp = float(g["reject_rate"].max() - g["reject_rate"].min())
        eo = float(g["tpr"].max() - g["tpr"].min())
        out["groups"][col] = {"by_group": {k: {m: round(float(v), 4) for m, v in r.items()} for k, r in g.iterrows()},
                              "dp_diff": round(dp, 4), "eo_diff": round(eo, 4)}
        worst_dp, worst_eo = max(worst_dp, dp), max(worst_eo, eo)
    out["worst_dp_diff"], out["worst_eo_diff"] = round(worst_dp, 4), round(worst_eo, 4)
    checks = {"auc": out["auc"] >= gate_cfg["min_auc"],
              "demographic_parity": worst_dp <= gate_cfg["max_dp_diff"],
              "equal_opportunity": worst_eo <= gate_cfg["max_eo_diff"]}
    out["checks"], out["passed"] = checks, all(checks.values())
    return out


def save_json(obj, path):
    Path(path).write_text(json.dumps(obj, indent=2, default=str))


def setup_mlflow():
    """Tracking DB and artifacts both live at the project root, whatever the working directory."""
    import mlflow
    mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")
    if mlflow.get_experiment_by_name("fairgate") is None:
        mlflow.create_experiment("fairgate", artifact_location=(ROOT / "mlruns").as_uri())
    mlflow.set_experiment("fairgate")


In [ ]:
%%writefile src/make_sample_data.py
"""Generate a SYNTHETIC file with the same columns as Kaggle's application_train.csv.

Used for CI and for a quick demo when the real data is not downloaded yet.
It deliberately contains age/gender related proxies so the fairness gate has
something real to catch. Results on this file are NOT results on Home Credit.
"""
import argparse
import numpy as np
import pandas as pd
from common import ROOT

def make(n=60000, seed=7):
    r = np.random.default_rng(seed)
    age = r.uniform(21, 69, n)
    male = r.random(n) < 0.34
    ext2 = np.clip(r.beta(4, 3, n) + 0.004 * (age - 45), 0, 1)
    ext3 = np.clip(r.beta(4, 3, n) + 0.006 * (age - 45), 0, 1)
    ext1 = np.clip(r.beta(3, 3, n) + 0.005 * (age - 45), 0, 1)
    income = np.exp(r.normal(11.9, 0.5, n)) * np.where(male, 1.15, 1.0)
    credit = income * r.uniform(1, 6, n)
    annuity = credit / r.uniform(10, 30, n)
    employed = -r.uniform(100, 9000, n)
    pension = age > 60
    employed[pension & (r.random(n) < 0.7)] = 365243
    logit = (-2.95 - 3.4 * (ext2 - 0.55) - 2.6 * (ext3 - 0.55) + 0.25 * (credit / income > 4)
             + 0.35 * male + 0.45 * (age < 30) - 0.25 * (age > 50) + r.normal(0, 0.4, n))
    target = (r.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
    ch = lambda opts, p=None: r.choice(opts, n, p=p)
    df = pd.DataFrame({
        "SK_ID_CURR": np.arange(100002, 100002 + n), "TARGET": target,
        "NAME_CONTRACT_TYPE": ch(["Cash loans", "Revolving loans"], [0.9, 0.1]),
        "CODE_GENDER": np.where(male, "M", "F"), "FLAG_OWN_CAR": ch(["Y", "N"], [0.34, 0.66]),
        "FLAG_OWN_REALTY": ch(["Y", "N"], [0.69, 0.31]), "CNT_CHILDREN": r.poisson(0.4, n),
        "AMT_INCOME_TOTAL": income.round(1), "AMT_CREDIT": credit.round(1), "AMT_ANNUITY": annuity.round(1),
        "AMT_GOODS_PRICE": (credit * 0.9).round(1),
        "NAME_INCOME_TYPE": np.where(pension, "Pensioner", ch(["Working", "Commercial associate", "State servant"], [0.65, 0.25, 0.1])),
        "NAME_EDUCATION_TYPE": ch(["Secondary / secondary special", "Higher education", "Incomplete higher"], [0.71, 0.24, 0.05]),
        "NAME_FAMILY_STATUS": ch(["Married", "Single / not married", "Civil marriage", "Separated", "Widow"], [0.64, 0.15, 0.1, 0.06, 0.05]),
        "NAME_HOUSING_TYPE": ch(["House / apartment", "With parents", "Municipal apartment", "Rented apartment"], [0.89, 0.05, 0.04, 0.02]),
        "REGION_POPULATION_RELATIVE": r.uniform(0.001, 0.07, n).round(6),
        "DAYS_BIRTH": (-age * 365.25).astype(int), "DAYS_EMPLOYED": employed.astype(int),
        "DAYS_REGISTRATION": -r.uniform(0, 15000, n).round(), "DAYS_ID_PUBLISH": -r.uniform(0, 6000, n).round(),
        "OWN_CAR_AGE": np.where(r.random(n) < 0.66, np.nan, r.integers(0, 30, n)),
        "OCCUPATION_TYPE": np.where(r.random(n) < 0.31, None, ch(["Laborers", "Sales staff", "Core staff", "Managers", "Drivers"])),
        "CNT_FAM_MEMBERS": r.integers(1, 5, n),
        "EXT_SOURCE_1": np.where(r.random(n) < 0.56, np.nan, ext1), "EXT_SOURCE_2": ext2,
        "EXT_SOURCE_3": np.where(r.random(n) < 0.2, np.nan, ext3),
    })
    return df

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--n", type=int, default=60000); a = ap.parse_args()
    out = ROOT / "data" / "raw" / "application_train.csv"
    df = make(a.n); df.to_csv(out, index=False)
    print(f"synthetic sample written: {out} rows={len(df)} default_rate={df.TARGET.mean():.3f}")


In [ ]:
%%writefile src/prepare.py
"""Stage 1: validate raw data, engineer features, split train/test."""
import sys
import pandas as pd
from sklearn.model_selection import train_test_split
from common import ROOT, PARAMS, PROC, REPORTS, TARGET, NUMERIC, CATEGORICAL, SENSITIVE_RAW, engineer, save_json

def validate(df):
    """Lightweight data checks (the role Great Expectations plays in the full design)."""
    errors = []
    needed = [TARGET] + NUMERIC + CATEGORICAL + SENSITIVE_RAW
    missing = [c for c in needed if c not in df.columns]
    if missing: errors.append(f"missing columns: {missing}")
    else:
        if not set(df[TARGET].unique()) <= {0, 1}: errors.append("TARGET must be 0/1")
        if (df["AMT_INCOME_TOTAL"] <= 0).any(): errors.append("AMT_INCOME_TOTAL must be > 0")
        if (df["AMT_CREDIT"] <= 0).any(): errors.append("AMT_CREDIT must be > 0")
        if (df["DAYS_BIRTH"] >= 0).any(): errors.append("DAYS_BIRTH must be negative")
        if df["EXT_SOURCE_2"].isna().mean() > 0.05: errors.append("EXT_SOURCE_2 null rate above 5%")
    return errors

def main():
    raw = ROOT / PARAMS["data"]["raw_path"]
    df = pd.read_csv(raw)
    df = df[df["CODE_GENDER"].isin(["M", "F"])]           # drops the 4 'XNA' rows in Home Credit
    errors = validate(df)
    save_json({"rows": len(df), "errors": errors}, REPORTS / "data_validation.json")
    if errors:
        print("DATA VALIDATION FAILED:", *errors, sep="\n  "); sys.exit(1)
    df = engineer(df)
    tr, te = train_test_split(df, test_size=PARAMS["data"]["test_size"], stratify=df[TARGET],
                              random_state=PARAMS["data"]["seed"])
    tr.to_parquet(PROC / "train.parquet"); te.to_parquet(PROC / "test.parquet")
    print(f"prepare: rows={len(df)} train={len(tr)} test={len(te)} default_rate={df[TARGET].mean():.3f}")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/train.py
"""Stage 2: train LightGBM, pick the decision threshold, log everything to MLflow."""
import joblib
import numpy as np
import pandas as pd
import mlflow
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from common import setup_mlflow, ROOT, PARAMS, PROC, MODELS, TARGET, CATEGORICAL, FEATURES, to_model_frame

setup_mlflow()

def main():
    tr = pd.read_parquet(PROC / "train.parquet"); te = pd.read_parquet(PROC / "test.parquet")
    cats = {c: sorted(tr[c].dropna().astype(str).unique().tolist()) for c in CATEGORICAL}
    Xtr, Xte = to_model_frame(tr, cats), to_model_frame(te, cats)
    p = PARAMS["train"]
    model = LGBMClassifier(n_estimators=p["n_estimators"], learning_rate=p["learning_rate"],
                           num_leaves=p["num_leaves"], subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.8, random_state=PARAMS["data"]["seed"], verbose=-1)
    with mlflow.start_run(run_name="train") as run:
        model.fit(Xtr, tr[TARGET])
        # Decision rule: reject the riskiest X% (threshold chosen on training scores).
        thr = float(np.quantile(model.predict_proba(Xtr)[:, 1], 1 - p["target_reject_rate"]))
        auc = roc_auc_score(te[TARGET], model.predict_proba(Xte)[:, 1])
        mlflow.log_params({**p, "n_features": len(FEATURES), "threshold": round(thr, 4)})
        mlflow.log_metric("test_auc", auc)
        mlflow.lightgbm.log_model(model, name="model")
        bundle = {"kind": "threshold", "model": model, "threshold": thr, "categories": cats,
                  "run_id": run.info.run_id, "version": "candidate"}
        joblib.dump(bundle, MODELS / "candidate.joblib")
        print(f"train: test AUC={auc:.4f} threshold={thr:.4f} run_id={run.info.run_id}")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/predictor.py
"""One place that turns a model bundle + applicant rows into scores, decisions and reason codes."""
import numpy as np
import pandas as pd
from common import FEATURES, to_model_frame

def group_key(df: pd.DataFrame) -> pd.Series:
    return df["gender"].astype(str) + "|" + df["age_band"].astype(str)

def predict(bundle: dict, df: pd.DataFrame, seed: int = 0):
    """Returns (risk score, decision) where decision 1 = REJECT."""
    X = to_model_frame(df, bundle["categories"])
    score = bundle["model"].predict_proba(X)[:, 1]
    if bundle["kind"] == "threshold":
        decision = (score >= bundle["threshold"]).astype(int)
    else:  # fairlearn ThresholdOptimizer: group-specific thresholds learned during mitigation
        decision = bundle["optimizer"].predict(X, sensitive_features=group_key(df), random_state=seed).astype(int)
    return score, decision

def reason_codes(bundle: dict, df: pd.DataFrame, top_k: int = 3):
    """TreeSHAP contributions computed natively by LightGBM (pred_contrib=True)."""
    X = to_model_frame(df, bundle["categories"])
    contrib = bundle["model"].booster_.predict(X, pred_contrib=True)[:, :-1]  # last column = base value
    out = []
    for row, vals in zip(X.itertuples(index=False), contrib):
        idx = np.argsort(-np.abs(vals))[:top_k]   # strongest drivers, either direction
        out.append([{"feature": FEATURES[i], "value": None if pd.isna(row[i]) else (row[i] if isinstance(row[i], str) else float(row[i])),
                     "impact": round(float(vals[i]), 4)} for i in idx])
    return out


In [ ]:
%%writefile src/gate.py
"""Stage 3: the FAIRNESS GATE. Exit code 0 = pass, 1 = blocked. CI uses the exit code."""
import argparse
import sys
import joblib
import pandas as pd
import mlflow
from common import setup_mlflow, ROOT, PARAMS, PROC, MODELS, REPORTS, TARGET, fairness_report, save_json
from predictor import predict

setup_mlflow()

def run_gate(model_name: str) -> dict:
    bundle = joblib.load(MODELS / f"{model_name}.joblib")
    te = pd.read_parquet(PROC / "test.parquet")
    score, decision = predict(bundle, te)
    rep = fairness_report(te[TARGET].values, decision, score, te, PARAMS["gate"])
    rep["model"] = model_name
    save_json(rep, REPORTS / f"gate_{model_name}.json")
    with mlflow.start_run(run_name=f"gate_{model_name}"):
        mlflow.log_metrics({"auc": rep["auc"], "worst_dp_diff": rep["worst_dp_diff"],
                            "worst_eo_diff": rep["worst_eo_diff"], "gate_passed": int(rep["passed"])})
        mlflow.log_artifact(str(REPORTS / f"gate_{model_name}.json"))
    status = "PASSED" if rep["passed"] else "FAILED"
    print(f"gate[{model_name}] {status}: AUC={rep['auc']:.4f} reject_rate={rep['reject_rate']:.3f} "
          f"DP gap={rep['worst_dp_diff']:.4f} (limit {PARAMS['gate']['max_dp_diff']}) "
          f"EO gap={rep['worst_eo_diff']:.4f} (limit {PARAMS['gate']['max_eo_diff']})")
    for col, g in rep["groups"].items():
        rates = ", ".join(f"{k}: {v['reject_rate']:.3f}" for k, v in g["by_group"].items())
        print(f"    reject rate by {col}: {rates}")
    return rep

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--model", default="candidate")
    rep = run_gate(ap.parse_args().model)
    sys.exit(0 if rep["passed"] else 1)


In [ ]:
%%writefile src/mitigate.py
"""Stage 4 (runs only when the gate fails): post-process the model with fairlearn ThresholdOptimizer.

It learns a separate decision threshold per group (gender x age band) so rejection rates
are equalised (demographic parity) while maximising balanced accuracy. The underlying
LightGBM model and its risk scores are unchanged; only the decision rule changes.
"""
import warnings
import joblib
import pandas as pd
import mlflow
from fairlearn.postprocessing import ThresholdOptimizer
from common import setup_mlflow, ROOT, PARAMS, PROC, MODELS, TARGET, to_model_frame
from predictor import group_key

warnings.filterwarnings("ignore")
setup_mlflow()

def main():
    b = joblib.load(MODELS / "candidate.joblib")
    tr = pd.read_parquet(PROC / "train.parquet")
    X = to_model_frame(tr, b["categories"])
    opt = ThresholdOptimizer(estimator=b["model"], constraints=PARAMS["gate"]["mitigation_constraint"],
                             objective="balanced_accuracy_score", prefit=True, predict_method="predict_proba")
    opt.fit(X, tr[TARGET], sensitive_features=group_key(tr))
    mitigated = {**b, "kind": "threshold_optimizer", "optimizer": opt, "version": "mitigated"}
    joblib.dump(mitigated, MODELS / "mitigated.joblib")
    with mlflow.start_run(run_name="mitigate"):
        mlflow.log_params({"method": "ThresholdOptimizer", "constraint": PARAMS["gate"]["mitigation_constraint"],
                           "objective": "balanced_accuracy_score", "groups": "gender x age_band",
                           "base_run_id": b["run_id"]})
    print("mitigate: ThresholdOptimizer fitted on gender x age_band groups -> models/mitigated.joblib")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/register.py
"""Stage 5: register ONLY a gate-passed model in the MLflow Model Registry and promote it."""
import argparse
import json
import shutil
import sys
import mlflow
from mlflow import MlflowClient
from common import ROOT, MODELS, REPORTS

mlflow.set_tracking_uri(f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}")
NAME = "fairgate-credit"

def main(model_name):
    rep = json.loads((REPORTS / f"gate_{model_name}.json").read_text())
    if not rep["passed"]:
        print(f"register: REFUSED, {model_name} did not pass the fairness gate"); sys.exit(1)
    import joblib
    bundle = joblib.load(MODELS / f"{model_name}.joblib")
    mv = mlflow.register_model(f"runs:/{bundle['run_id']}/model", NAME)
    client = MlflowClient()
    client.set_registered_model_alias(NAME, "production", mv.version)
    for k, v in {"gate_passed": "true", "decision_rule": bundle["kind"],
                 "dp_gap": str(rep["worst_dp_diff"]), "eo_gap": str(rep["worst_eo_diff"])}.items():
        client.set_model_version_tag(NAME, mv.version, k, v)
    shutil.copy(MODELS / f"{model_name}.joblib", MODELS / "production.joblib")
    print(f"register: {NAME} v{mv.version} -> alias 'production' (decision rule: {bundle['kind']})")

if __name__ == "__main__":
    ap = argparse.ArgumentParser(); ap.add_argument("--model", default="candidate")
    main(ap.parse_args().model)


In [ ]:
%%writefile src/monitor.py
"""Stage 6: simulate production traffic and watch for DATA drift (PSI) and FAIRNESS drift.

Scenario: a new acquisition channel brings in more applicants under 30, and those
applicants have progressively weaker bureau scores (EXT_SOURCE). Labels are not
available in production yet, so fairness is tracked with the label-free metric:
the gap in rejection rate between groups (demographic parity).
"""
import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from common import PARAMS, PROC, MODELS, REPORTS, NUMERIC, ENGINEERED, save_json
from predictor import predict

def psi(ref, cur, bins=10):
    ref, cur = pd.Series(ref).dropna(), pd.Series(cur).dropna()
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3: return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    r = np.histogram(ref, edges)[0] / len(ref); c = np.histogram(cur, edges)[0] / len(cur)
    r, c = np.clip(r, 1e-4, None), np.clip(c, 1e-4, None)
    return float(np.sum((c - r) * np.log(c / r)))

def make_batch(te, k, rng, cfg):
    young_share = 0.2 + 0.05 * k
    young, rest = te[te.age_band == "<30"], te[te.age_band != "<30"]
    n_y = int(cfg["batch_size"] * young_share)
    b = pd.concat([young.sample(n_y, replace=True, random_state=int(rng.integers(1e9))),
                   rest.sample(cfg["batch_size"] - n_y, replace=True, random_state=int(rng.integers(1e9)))])
    m = b.age_band == "<30"
    for c in ("EXT_SOURCE_2", "EXT_SOURCE_3"):
        b.loc[m, c] = (b.loc[m, c] - 0.035 * k).clip(0, 1)
    return b

def main():
    cfg = PARAMS["monitor"]; rng = np.random.default_rng(1)
    bundle = joblib.load(MODELS / "production.joblib")
    ref = pd.read_parquet(PROC / "train.parquet"); te = pd.read_parquet(PROC / "test.parquet")
    feats = NUMERIC + ENGINEERED
    rows = []
    for k in range(cfg["n_batches"]):
        b = make_batch(te, k, rng, cfg)
        _, dec = predict(bundle, b, seed=k)
        b = b.assign(rejected=dec)
        rate = {col: b.groupby(col)["rejected"].mean() for col in ("gender", "age_band")}
        gap = max(float(v.max() - v.min()) for v in rate.values())
        psis = {f: psi(ref[f], b[f]) for f in feats}
        top = max(psis, key=psis.get)
        rows.append({"batch": k + 1, "share_under_30": round(float((b.age_band == "<30").mean()), 3),
                     "reject_rate": round(float(b.rejected.mean()), 4),
                     "reject_rate_by_age": {g: round(float(v), 4) for g, v in rate["age_band"].items()},
                     "fairness_gap": round(gap, 4), "max_psi_feature": top, "max_psi": round(psis[top], 4),
                     "fairness_alert": gap > cfg["fair_limit"], "data_drift_alert": psis[top] > cfg["psi_limit"]})
        r = rows[-1]
        flag = ("RETRAIN TRIGGERED" if (r["fairness_alert"] or r["data_drift_alert"]) else "ok")
        print(f"batch {r['batch']}: under30={r['share_under_30']:.2f} reject={r['reject_rate']:.3f} "
              f"fairness gap={r['fairness_gap']:.3f} max PSI={r['max_psi']:.3f} ({top}) -> {flag}")
    first = next((r["batch"] for r in rows if r["fairness_alert"] or r["data_drift_alert"]), None)
    save_json({"limits": cfg, "batches": rows, "retrain_triggered_at_batch": first}, REPORTS / "monitoring.json")

    x = [r["batch"] for r in rows]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(x, [r["fairness_gap"] for r in rows], "o-", color="#8B1E2E", label="rejection-rate gap")
    ax[0].axhline(cfg["fair_limit"], ls="--", color="grey", label=f"limit {cfg['fair_limit']}")
    ax[0].set(title="Fairness drift (label-free)", xlabel="production batch", ylabel="max gap between groups"); ax[0].legend()
    ax[1].plot(x, [r["max_psi"] for r in rows], "o-", color="#1F3864", label="max PSI across features")
    ax[1].axhline(cfg["psi_limit"], ls="--", color="grey", label=f"limit {cfg['psi_limit']}")
    ax[1].set(title="Data drift (PSI)", xlabel="production batch", ylabel="PSI"); ax[1].legend()
    if first:
        for a in ax: a.axvline(first, color="#C9781E", alpha=0.6)
    fig.tight_layout(); fig.savefig(REPORTS / "monitoring.png", dpi=140)
    print(f"monitor: retrain trigger first fired at batch {first}" if first else "monitor: no drift alerts")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_pipeline.py
"""Runs the whole FairGate pipeline end to end (same steps CI runs).

prepare -> train -> GATE(candidate) --pass--> register -> monitor -> report
                          |fail
                          v
                    mitigate -> GATE(mitigated) --pass--> register
                                     |fail -> pipeline stops, nothing is deployed
"""
import base64
import json
import subprocess
import sys
from pathlib import Path

ROOT = Path(__file__).parent
SRC = ROOT / "src"; REP = ROOT / "reports"

def step(name, *args, allow_fail=False):
    print(f"\n=== {name} ===", flush=True)
    r = subprocess.run([sys.executable, str(SRC / f"{name}.py"), *args], cwd=SRC)
    if r.returncode != 0 and not allow_fail:
        print(f"\nPIPELINE STOPPED at {name}"); sys.exit(r.returncode)
    return r.returncode == 0

def report():
    gates = {m: json.loads((REP / f"gate_{m}.json").read_text()) for m in ("candidate", "mitigated")
             if (REP / f"gate_{m}.json").exists()}
    mon = json.loads((REP / "monitoring.json").read_text())
    img = base64.b64encode((REP / "monitoring.png").read_bytes()).decode()
    def gate_html(m, g):
        badge = "<b style='color:#1a7f37'>PASSED</b>" if g["passed"] else "<b style='color:#c62828'>FAILED (blocked)</b>"
        rows = "".join(f"<tr><td>{col}</td><td>{grp}</td><td>{v['reject_rate']:.3f}</td><td>{v['tpr']:.3f}</td></tr>"
                       for col, d in g["groups"].items() for grp, v in d["by_group"].items())
        return (f"<h3>Model: {m} &nbsp; {badge}</h3><p>AUC {g['auc']:.4f} &middot; overall reject rate {g['reject_rate']:.3f} "
                f"&middot; demographic parity gap {g['worst_dp_diff']:.3f} &middot; equal opportunity gap {g['worst_eo_diff']:.3f}</p>"
                f"<table><tr><th>Attribute</th><th>Group</th><th>Reject rate</th><th>TPR (defaulters caught)</th></tr>{rows}</table>")
    mrows = "".join(f"<tr><td>{r['batch']}</td><td>{r['share_under_30']:.2f}</td><td>{r['reject_rate']:.3f}</td>"
                    f"<td>{r['fairness_gap']:.3f}</td><td>{r['max_psi']:.3f} ({r['max_psi_feature']})</td>"
                    f"<td>{'RETRAIN' if r['fairness_alert'] or r['data_drift_alert'] else 'ok'}</td></tr>" for r in mon["batches"])
    html = f"""<!doctype html><meta charset=utf-8><title>FairGate report</title>
<style>body{{font-family:Arial;max-width:1000px;margin:30px auto;color:#222}}h1{{color:#8B1E2E}}table{{border-collapse:collapse;margin:8px 0 18px}}
td,th{{border:1px solid #ccc;padding:5px 10px}}th{{background:#8B1E2E;color:#fff}}</style>
<h1>FairGate pipeline report</h1>
<h2>1. Fairness gate</h2>{''.join(gate_html(m, g) for m, g in gates.items())}
<h2>2. Production monitoring (simulated)</h2><img src="data:image/png;base64,{img}" width="100%">
<table><tr><th>Batch</th><th>Share under 30</th><th>Reject rate</th><th>Fairness gap</th><th>Max PSI</th><th>Status</th></tr>{mrows}</table>
<p>Retrain first triggered at batch: <b>{mon['retrain_triggered_at_batch']}</b></p>"""
    (REP / "fairgate_report.html").write_text(html)
    print(f"\nreport written: {REP / 'fairgate_report.html'}")

if __name__ == "__main__":
    step("prepare"); step("train")
    if step("gate", "--model", "candidate", allow_fail=True):
        step("register", "--model", "candidate")
    else:
        print("\n>>> candidate BLOCKED by fairness gate -> running mitigation")
        step("mitigate"); step("gate", "--model", "mitigated"); step("register", "--model", "mitigated")
    step("monitor"); report()


In [ ]:
%%writefile app/__init__.py
# package marker

In [ ]:
%%writefile app/main.py
"""FairGate scoring API: risk score + approve/reject decision + top 3 SHAP reason codes."""
import sys
from pathlib import Path
from typing import Optional
import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
import json
from pydantic import BaseModel

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from common import engineer  # noqa: E402
from predictor import predict, reason_codes  # noqa: E402

app = FastAPI(title="FairGate credit scoring API", version="1.0")
BUNDLE = None

def bundle():
    global BUNDLE
    if BUNDLE is None:
        path = ROOT / "models" / "production.joblib"
        if not path.exists():
            raise HTTPException(503, "No production model. Run the pipeline first (python run_pipeline.py).")
        BUNDLE = joblib.load(path)
    return BUNDLE

class Applicant(BaseModel):
    CODE_GENDER: str
    DAYS_BIRTH: int
    AMT_INCOME_TOTAL: float
    AMT_CREDIT: float
    AMT_ANNUITY: float
    AMT_GOODS_PRICE: float
    DAYS_EMPLOYED: int
    EXT_SOURCE_1: Optional[float] = None
    EXT_SOURCE_2: Optional[float] = None
    EXT_SOURCE_3: Optional[float] = None
    REGION_POPULATION_RELATIVE: float = 0.02
    CNT_CHILDREN: int = 0
    CNT_FAM_MEMBERS: float = 2
    DAYS_REGISTRATION: float = -4000
    DAYS_ID_PUBLISH: float = -3000
    OWN_CAR_AGE: Optional[float] = None
    NAME_CONTRACT_TYPE: str = "Cash loans"
    NAME_INCOME_TYPE: str = "Working"
    NAME_EDUCATION_TYPE: str = "Secondary / secondary special"
    NAME_FAMILY_STATUS: str = "Married"
    NAME_HOUSING_TYPE: str = "House / apartment"
    OCCUPATION_TYPE: Optional[str] = None
    FLAG_OWN_CAR: str = "N"
    FLAG_OWN_REALTY: str = "Y"

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": (ROOT / "models" / "production.joblib").exists()}

@app.get("/model")
def model_info():
    b = bundle()
    return {"version": b["version"], "decision_rule": b["kind"], "mlflow_run_id": b["run_id"]}

@app.post("/predict")
def score(a: Applicant):
    b = bundle()
    df = engineer(pd.DataFrame([a.model_dump()]))
    risk, decision = predict(b, df)
    return {"default_risk": round(float(risk[0]), 4),
            "decision": "REJECT" if decision[0] == 1 else "APPROVE",
            "reason_codes": reason_codes(b, df)[0],
            "model_version": b["version"], "decision_rule": b["kind"]}


REPORTS = ROOT / "reports"

@app.get("/gate")
def gate_results():
    """Gate reports for every model that went through the fairness gate."""
    out = {}
    for m in ("candidate", "mitigated"):
        f = REPORTS / f"gate_{m}.json"
        if f.exists():
            g = json.loads(f.read_text())
            out[m] = {k: g[k] for k in ("passed", "auc", "reject_rate", "worst_dp_diff", "worst_eo_diff", "groups")}
    return out

@app.get("/monitoring.png")
def monitoring_chart():
    f = REPORTS / "monitoring.png"
    if not f.exists():
        raise HTTPException(404, "Run the pipeline to create the monitoring chart.")
    return FileResponse(f)

@app.get("/report", response_class=HTMLResponse)
def full_report():
    f = REPORTS / "fairgate_report.html"
    return f.read_text() if f.exists() else "<p>Run the pipeline to create the report.</p>"

@app.get("/", response_class=HTMLResponse)
def home():
    return (ROOT / "app" / "index.html").read_text()


In [ ]:
%%writefile tests/test_fairgate.py
"""Run after the pipeline: pytest -q"""
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from fastapi.testclient import TestClient

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src")); sys.path.insert(0, str(ROOT))
from common import fairness_report  # noqa: E402
from app.main import app  # noqa: E402

GATE = {"sensitive": ["g"], "max_dp_diff": 0.05, "max_eo_diff": 0.10, "min_auc": 0.5}

def test_gate_blocks_biased_decisions():
    y = np.array([1, 0, 1, 0, 1, 0, 1, 0]); score = np.linspace(0, 1, 8)
    sens = pd.DataFrame({"g": ["a"] * 4 + ["b"] * 4})
    biased = np.array([1, 1, 1, 1, 0, 0, 0, 0])          # rejects everyone in group a, nobody in b
    assert not fairness_report(y, biased, score, sens, GATE)["passed"]

def test_gate_passes_equal_decisions():
    y = np.array([1, 0, 1, 0, 1, 0, 1, 0]); score = y * 0.9 + 0.05
    sens = pd.DataFrame({"g": ["a"] * 4 + ["b"] * 4})
    assert fairness_report(y, y.copy(), score, sens, GATE)["passed"]

def test_api_returns_decision_and_three_reason_codes():
    c = TestClient(app)
    r = c.post("/predict", json={"CODE_GENDER": "F", "DAYS_BIRTH": -9500, "AMT_INCOME_TOTAL": 150000,
                                 "AMT_CREDIT": 700000, "AMT_ANNUITY": 32000, "AMT_GOODS_PRICE": 630000,
                                 "DAYS_EMPLOYED": -800, "EXT_SOURCE_2": 0.3, "EXT_SOURCE_3": 0.25})
    assert r.status_code == 200
    body = r.json()
    assert body["decision"] in ("APPROVE", "REJECT")
    assert len(body["reason_codes"]) == 3


## 3. Get the data
**Option A (default): synthetic sample** with the same columns as Home Credit. Good for demo; not real results.

**Option B: real Home Credit data.** Set `USE_REAL_DATA = True`, then upload your `kaggle.json`
(Kaggle → Settings → API → Create New Token). You must click *Join Competition / accept rules* once on
https://www.kaggle.com/c/home-credit-default-risk first.

In [ ]:
USE_REAL_DATA = False

if USE_REAL_DATA:
    from google.colab import files
    files.upload()                      # choose kaggle.json
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !kaggle competitions download -c home-credit-default-risk -f application_train.csv -p data/raw
    !cd data/raw && (unzip -o application_train.csv.zip || true)
else:
    !cd src && python make_sample_data.py

import pandas as pd
raw = pd.read_csv('data/raw/application_train.csv')
print(raw.shape, '| default rate:', round(raw.TARGET.mean(), 3))
raw[['CODE_GENDER','DAYS_BIRTH','AMT_INCOME_TOTAL','AMT_CREDIT','EXT_SOURCE_2','EXT_SOURCE_3','TARGET']].head()

## 4. Run the full pipeline
`prepare → train → FAIRNESS GATE → (mitigate → gate again) → register → monitor`

Watch the first gate **fail** and block the model, mitigation run, and the second gate **pass**.

In [ ]:
!python run_pipeline.py 2>&1 | grep -v -iE 'warning|INFO mlflow|alembic|context impl|will assume'

## 5. Gate results: before vs after mitigation

In [ ]:
import json
rows = []
for m in ['candidate', 'mitigated']:
    try: g = json.load(open(f'reports/gate_{m}.json'))
    except FileNotFoundError: continue
    for attr, d in g['groups'].items():
        for grp, v in d['by_group'].items():
            rows.append({'model': m, 'attribute': attr, 'group': grp,
                         'reject_rate': v['reject_rate'], 'TPR (defaulters caught)': v['tpr']})
    print(f"{m:10s} | gate {'PASSED' if g['passed'] else 'FAILED'} | AUC {g['auc']:.4f} | "
          f"DP gap {g['worst_dp_diff']:.3f} | EO gap {g['worst_eo_diff']:.3f} | reject rate {g['reject_rate']:.3f}")
pd.DataFrame(rows).pivot_table(index=['attribute','group'], columns='model', values='reject_rate')

In [ ]:
import matplotlib.pyplot as plt
df = pd.DataFrame(rows); ab = df[df.attribute == 'age_band']
fig, ax = plt.subplots(figsize=(7, 3.5))
for i, m in enumerate(ab.model.unique()):
    s = ab[ab.model == m].set_index('group').reindex(['<30', '30-50', '>50'])
    ax.bar([x + i * 0.38 for x in range(3)], s.reject_rate, 0.38, label=m)
ax.set_xticks([0.19, 1.19, 2.19]); ax.set_xticklabels(['<30', '30-50', '>50'])
ax.set_ylabel('rejection rate'); ax.set_title('Rejection rate by age band: before vs after the gate'); ax.legend(); plt.show()

## 6. Production monitoring: fairness drift vs data drift
A new channel brings more under-30 applicants with weaker bureau scores. The fairness alarm fires well before PSI does.

In [ ]:
from IPython.display import Image, display
display(Image('reports/monitoring.png'))
mon = json.load(open('reports/monitoring.json'))
pd.DataFrame(mon['batches'])[['batch','share_under_30','reject_rate','fairness_gap','max_psi','max_psi_feature','fairness_alert','data_drift_alert']]

## 7. MLflow: experiment runs and the model registry

In [ ]:
import mlflow
from mlflow import MlflowClient
mlflow.set_tracking_uri('sqlite:////content/fairgate/mlflow.db')
runs = mlflow.search_runs(experiment_names=['fairgate'])
cols = [c for c in ['tags.mlflow.runName','metrics.test_auc','metrics.auc','metrics.worst_dp_diff','metrics.worst_eo_diff','metrics.gate_passed'] if c in runs]
display(runs[cols])
c = MlflowClient()
prod = c.get_model_version_by_alias('fairgate-credit', 'production')
print(f"Registry: fairgate-credit v{prod.version} has alias 'production' | tags {prod.tags}")

## 8. Explainable serving: call the FastAPI endpoint
Each decision returns its top 3 reason codes (TreeSHAP).

In [ ]:
import sys; sys.path.insert(0, '/content/fairgate')
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app)
print(client.get('/model').json())
applicant = {'CODE_GENDER': 'M', 'DAYS_BIRTH': -9100, 'AMT_INCOME_TOTAL': 135000, 'AMT_CREDIT': 900000,
             'AMT_ANNUITY': 40000, 'AMT_GOODS_PRICE': 810000, 'DAYS_EMPLOYED': -400,
             'EXT_SOURCE_2': 0.21, 'EXT_SOURCE_3': 0.18}
r = client.post('/predict', json=applicant).json()
print(f"\nDecision: {r['decision']}  (default risk {r['default_risk']})")
for rc in r['reason_codes']:
    print(f"  {rc['feature']:22s} value={rc['value'] if isinstance(rc['value'], str) else round(rc['value'], 3)}  impact={rc['impact']:+.3f}")

## 9. Tests (same ones CI runs)

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3

## 10. Deploy: live Gradio app with a public link
Running this cell gives a public `https://….gradio.live` URL anyone can open.
The link works only while this Colab session is running, so start it shortly before you present.


In [ ]:
import gradio as gr, joblib, json, sys
sys.path.insert(0, '/content/fairgate/src')
from common import engineer
from predictor import predict, reason_codes

BUNDLE = joblib.load('/content/fairgate/models/production.joblib')
LABEL = {'EXT_SOURCE_1': 'Bureau score 1', 'EXT_SOURCE_2': 'Bureau score 2', 'EXT_SOURCE_3': 'Bureau score 3',
         'credit_income_ratio': 'Loan to income ratio', 'annuity_income_ratio': 'Repayment to income ratio',
         'credit_term': 'Repayment to loan ratio', 'goods_credit_ratio': 'Goods price to loan ratio',
         'DAYS_EMPLOYED': 'Time employed', 'DAYS_REGISTRATION': 'Time since registration',
         'DAYS_ID_PUBLISH': 'Time since ID issued', 'AMT_CREDIT': 'Loan amount', 'AMT_INCOME_TOTAL': 'Income'}

def score(gender, age, income, loan, repayment, years_employed, bureau2, bureau3):
    row = {'CODE_GENDER': gender, 'DAYS_BIRTH': int(-age * 365.25), 'AMT_INCOME_TOTAL': income,
           'AMT_CREDIT': loan, 'AMT_ANNUITY': repayment, 'AMT_GOODS_PRICE': loan * 0.9,
           'DAYS_EMPLOYED': int(-years_employed * 365.25), 'EXT_SOURCE_2': bureau2, 'EXT_SOURCE_3': bureau3,
           'REGION_POPULATION_RELATIVE': 0.02, 'CNT_CHILDREN': 0, 'CNT_FAM_MEMBERS': 2, 'DAYS_REGISTRATION': -4000,
           'DAYS_ID_PUBLISH': -3000, 'NAME_CONTRACT_TYPE': 'Cash loans', 'NAME_INCOME_TYPE': 'Working',
           'NAME_EDUCATION_TYPE': 'Secondary / secondary special', 'NAME_FAMILY_STATUS': 'Married',
           'NAME_HOUSING_TYPE': 'House / apartment', 'OCCUPATION_TYPE': None, 'FLAG_OWN_CAR': 'N', 'FLAG_OWN_REALTY': 'Y',
           'EXT_SOURCE_1': None, 'OWN_CAR_AGE': None}
    df = engineer(pd.DataFrame([row]))
    risk, dec = predict(BUNDLE, df)
    verdict = 'REJECTED' if dec[0] == 1 else 'APPROVED'
    reasons = pd.DataFrame([{'Reason': LABEL.get(r['feature'], r['feature']),
                             'Effect on risk': 'raises' if r['impact'] > 0 else 'lowers',
                             'Impact (SHAP)': round(r['impact'], 3)} for r in reason_codes(BUNDLE, df)[0]])
    return f"## {verdict}\n**Default risk: {risk[0]*100:.1f}%** · model `{BUNDLE['version']}` ({BUNDLE['kind']})", reasons

def gate_table():
    out = []
    for m in ('candidate', 'mitigated'):
        try: g = json.load(open(f'/content/fairgate/reports/gate_{m}.json'))
        except FileNotFoundError: continue
        ab = g['groups']['age_band']['by_group']
        out.append({'Model': 'First model' if m == 'candidate' else 'After mitigation',
                    'Gate': 'PASSED' if g['passed'] else 'BLOCKED', 'AUC': round(g['auc'], 3),
                    'Parity gap': g['worst_dp_diff'], 'Equal opp. gap': g['worst_eo_diff'],
                    'Reject rate <30': ab['<30']['reject_rate'], 'Reject rate >50': ab['>50']['reject_rate']})
    return pd.DataFrame(out)

mon = pd.DataFrame(json.load(open('/content/fairgate/reports/monitoring.json'))['batches'])[
    ['batch', 'share_under_30', 'reject_rate', 'fairness_gap', 'max_psi', 'fairness_alert', 'data_drift_alert']]

with gr.Blocks(title='FairGate') as demo:
    gr.Markdown('# FairGate\nCredit default scoring where a model only goes live after passing a bias check, '
                'every decision explains itself, and fairness is monitored in production.')
    with gr.Tab('Score an applicant'):
        gr.Markdown('Gender and age are **not** model inputs. They are used only to check and correct bias.')
        with gr.Row():
            with gr.Column():
                gender = gr.Radio(['F', 'M'], value='M', label='Gender')
                age = gr.Slider(18, 75, value=26, step=1, label='Age')
                income = gr.Number(value=135000, label='Annual income')
                loan = gr.Number(value=900000, label='Loan amount')
                repay = gr.Number(value=40000, label='Yearly repayment')
                emp = gr.Slider(0, 40, value=1, step=0.5, label='Years employed')
                b2 = gr.Slider(0, 1, value=0.21, step=0.01, label='Bureau score 2')
                b3 = gr.Slider(0, 1, value=0.18, step=0.01, label='Bureau score 3')
                btn = gr.Button('Score applicant', variant='primary')
            with gr.Column():
                verdict = gr.Markdown()
                reasons = gr.Dataframe(label='Strongest reasons (TreeSHAP)')
        inputs = [gender, age, income, loan, repay, emp, b2, b3]
        btn.click(score, inputs, [verdict, reasons])
        gr.Examples([['M', 26, 135000, 900000, 40000, 1, 0.21, 0.18],
                     ['F', 47, 220000, 450000, 22000, 12, 0.72, 0.68]], inputs, label='Try these applicants')
    with gr.Tab('Release gate'):
        gr.Markdown('Every candidate model is checked for bias before it can be registered. '
                    'Limits: parity gap 0.05, equal opportunity gap 0.10, AUC at least 0.70.')
        gr.Dataframe(gate_table())
    with gr.Tab('Production monitoring'):
        gr.Markdown('More young applicants with weaker bureau scores arrive each batch. '
                    'The fairness alarm fires at batch 3, well before data drift (PSI) crosses its limit.')
        gr.Image('/content/fairgate/reports/monitoring.png', show_label=False)
        gr.Dataframe(mon)

demo.launch(share=True)


## 11. Download the full HTML report

In [ ]:
from IPython.display import HTML
display(HTML(open('reports/fairgate_report.html').read()))
try:
    from google.colab import files; files.download('reports/fairgate_report.html')
except ImportError:
    pass